# Green's Functions: Resolvent Response Solver

This notebook computes a finite-system Green's function response. For a Hamiltonian `H`, the retarded Green's function is `(omega + i eta - H)^-1`. The imaginary part of a matrix element gives a broadened spectral response.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qsvt.spectral import apply_polynomial_to_hermitian, eigh_hermitian
from qsvt.hamiltonians import tight_binding_chain
from qsvt.matrix_functions import design_resolvent_polynomials

H = tight_binding_chain(5)
probe = np.zeros(5)
probe[2] = 1.0
eta = 0.35
omegas = np.linspace(-2.2, 2.2, 120)
evals, _ = eigh_hermitian(H)
scale = np.max(np.abs(evals))
A = H / scale

exact_response = []
poly_response = []
for omega in omegas:
    exact_G = np.linalg.inv((omega + 1j * eta) * np.eye(5) - H)
    exact_response.append(-np.imag(np.vdot(probe, exact_G @ probe)) / np.pi)

    _, imag_coeffs = design_resolvent_polynomials(
        omega=omega, eta=eta, scale=scale, degree=34
    )
    imag_G = apply_polynomial_to_hermitian(A, imag_coeffs)
    poly_response.append(-np.vdot(probe, imag_G @ probe).real / np.pi)

response_error = np.linalg.norm(
    np.array(poly_response) - np.array(exact_response)
) / np.linalg.norm(exact_response)
response_error

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(omegas, exact_response, label="exact local spectral function")
plt.plot(omegas, poly_response, "--", label="inverse-polynomial approximation")
plt.xlabel("frequency omega")
plt.ylabel("A(omega)")
plt.title(f"Green's function response, relative curve error={response_error:.2e}")
plt.legend()
plt.show()